# 01 — Text Extraction & Cleaning

Extract text from all resume sources and produce a single clean JSON for the rest of the pipeline.

**Inputs:**
| Source | Location | Format |
|--------|----------|--------|
| DS3 Members | `data/ds3/member_resumes/` (Google Drive links in `member.txt`) | PDF (downloaded) |
| Discord | `data/discord/pdfs/` and `data/discord/images/` | PDF + images |
| Reddit | `data/reddit/resumes/files/` | PDF + images |
| Kaggle | `data/resume-dataset/Resume/Resume.csv` | CSV (text already extracted) |

**Output:** `data/processed/resumes_extracted.json`  
A list of records, each with `{filename, source, text, file_path, word_count, metadata}`

In [ ]:
import os
import json
import re
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List, Dict
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import pdfplumber
from PIL import Image
import pytesseract

PROJECT_ROOT = Path("../").resolve()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = PROCESSED_DIR / "resumes_extracted.json"

print(f"Project root : {PROJECT_ROOT}")
print(f"Data dir     : {DATA_DIR}")
print(f"Output path  : {OUTPUT_PATH}")

## 1 — Extraction helpers

In [ ]:
def extract_text_pdf(file_path: Path) -> str:
    """Extract text from a PDF using pdfplumber."""
    try:
        with pdfplumber.open(file_path) as pdf:
            pages = [page.extract_text() or "" for page in pdf.pages]
            return "\n".join(pages).strip()
    except Exception as e:
        print(f"  [WARN] PDF extraction failed for {file_path.name}: {e}")
        return ""


def extract_text_image(file_path: Path) -> str:
    """OCR an image file using pytesseract."""
    try:
        image = Image.open(file_path)
        return pytesseract.image_to_string(image).strip()
    except Exception as e:
        print(f"  [WARN] OCR failed for {file_path.name}: {e}")
        return ""


def extract_text(file_path: Path) -> str:
    """Route to the correct extractor based on file extension."""
    ext = file_path.suffix.lower()
    if ext == ".pdf":
        return extract_text_pdf(file_path)
    elif ext in {".jpg", ".jpeg", ".png", ".gif", ".webp"}:
        return extract_text_image(file_path)
    elif ext == ".txt":
        return file_path.read_text(encoding="utf-8", errors="replace").strip()
    else:
        print(f"  [SKIP] Unsupported extension: {ext} ({file_path.name})")
        return ""


print("Extraction helpers loaded.")

## 2 — Text cleaning

In [ ]:
MIN_TEXT_LENGTH = 100  # chars — anything shorter is likely a failed extraction


def clean_text(raw: str) -> str:
    """Normalize whitespace, strip non-printable chars, collapse blank lines."""
    text = raw.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[^\S\n]+", " ", text)          # collapse horizontal whitespace
    text = re.sub(r"\n{3,}", "\n\n", text)          # max 2 consecutive newlines
    text = re.sub(r"[^\x20-\x7E\n]", "", text)      # drop non-printable ASCII
    return text.strip()


print("Cleaning function loaded.")

## 3 — Process each data source

Each helper scans a directory, extracts + cleans text, and returns a list of resume records.

In [ ]:
def process_folder(folder: Path, source: str, limit: int | None = None) -> List[Dict]:
    """Extract text from every supported file in *folder*."""
    if not folder.exists():
        print(f"[SKIP] Folder not found: {folder}")
        return []

    files = sorted(f for f in folder.iterdir() if f.is_file() and not f.name.startswith("."))
    if limit:
        files = files[:limit]

    records = []
    for fp in tqdm(files, desc=source):
        raw = extract_text(fp)
        text = clean_text(raw)
        if len(text) < MIN_TEXT_LENGTH:
            continue
        records.append({
            "filename": fp.name,
            "source": source,
            "text": text,
            "file_path": str(fp),
            "word_count": len(text.split()),
        })

    print(f"  -> {len(records)} / {len(files)} files yielded usable text")
    return records


print("Folder processor loaded.")

### 3a — DS3 Member Resumes

`member.txt` contains Google Drive links. These PDFs need to be **downloaded first** into
`data/ds3/member_resumes/pdfs/` before we can extract text.  
The cell below assumes the PDFs have already been downloaded (see `data/ds3/ds3.ipynb` for the download script).  
If the PDFs haven't been downloaded yet, run that notebook first.

In [ ]:
# -------------------------------------------------------------------
# DS3 Member Resumes
# -------------------------------------------------------------------
# TODO: Update this path once the Google Drive download script places
#       PDFs in a local folder. Currently member.txt only has links.
#       Expected path after download: data/ds3/member_resumes/pdfs/
# -------------------------------------------------------------------

DS3_PDF_DIR = DATA_DIR / "ds3" / "member_resumes" / "pdfs"

ds3_records = process_folder(DS3_PDF_DIR, source="ds3_members")

### 3b — Discord Resumes (from Dropbox)

PDFs live in `data/discord/pdfs/`, images in `data/discord/images/`.  
Images will go through OCR (pytesseract).

In [ ]:
discord_pdf_records = process_folder(DATA_DIR / "discord" / "pdfs", source="discord")
discord_img_records = process_folder(DATA_DIR / "discord" / "images", source="discord")

discord_records = discord_pdf_records + discord_img_records
print(f"\nDiscord total: {len(discord_records)} resumes")

### 3c — Reddit Resumes

In [ ]:
reddit_records = process_folder(DATA_DIR / "reddit" / "resumes" / "files", source="reddit")

### 3d — Kaggle Resume Dataset

The Kaggle dataset is a CSV where the `Resume_str` column already contains extracted text.  
We filter to the ENGINEERING category to keep it relevant.

In [ ]:
KAGGLE_CSV = DATA_DIR / "resume-dataset" / "Resume" / "Resume.csv"

kaggle_records = []
if KAGGLE_CSV.exists():
    kaggle_df = pd.read_csv(KAGGLE_CSV)
    print(f"Kaggle CSV loaded: {len(kaggle_df)} rows")
    print(f"Categories: {kaggle_df['Category'].unique().tolist()}")

    # TODO: Adjust the category filter as needed. Using a broad set for
    #       more training data; narrow to ["ENGINEERING"] if desired.
    TARGET_CATEGORIES = [
        "ENGINEERING",
        "INFORMATION-TECHNOLOGY",
        "BUSINESS-DEVELOPMENT",
    ]
    filtered = kaggle_df[kaggle_df["Category"].isin(TARGET_CATEGORIES)]
    print(f"Filtered to {len(filtered)} rows in {TARGET_CATEGORIES}")

    for idx, row in filtered.iterrows():
        raw = str(row.get("Resume_str", ""))
        text = clean_text(raw)
        if len(text) < MIN_TEXT_LENGTH:
            continue
        kaggle_records.append({
            "filename": f"kaggle_{idx}.txt",
            "source": "kaggle",
            "text": text,
            "file_path": str(KAGGLE_CSV),
            "word_count": len(text.split()),
        })

    print(f"  -> {len(kaggle_records)} Kaggle resumes kept")
else:
    print(f"[SKIP] Kaggle CSV not found at {KAGGLE_CSV}")

## 4 — Enrich DS3 resumes with member metadata

Join info from `members.csv` (name, major, graduation year, links) onto the DS3 records
so it travels with the resume through the rest of the pipeline.

In [ ]:
MEMBERS_CSV = DATA_DIR / "ds3" / "member_resumes" / "members.csv"

members_df = None
if MEMBERS_CSV.exists():
    members_df = pd.read_csv(MEMBERS_CSV)
    print(f"Loaded members.csv: {len(members_df)} rows")
    print(f"Columns: {members_df.columns.tolist()}")
else:
    print(f"[SKIP] members.csv not found at {MEMBERS_CSV}")

In [ ]:
def enrich_ds3_records(records: List[Dict], members_df: pd.DataFrame) -> List[Dict]:
    """Fuzzy-match DS3 resume filenames to rows in members.csv and attach metadata."""
    if members_df is None or members_df.empty:
        return records

    for rec in records:
        stem = Path(rec["filename"]).stem.replace("_", " ").replace("-", " ").lower()
        for _, row in members_df.iterrows():
            name = str(row.get("Full Name", "")).lower()
            if name and name in stem:
                rec["metadata"] = {
                    "full_name": row.get("Full Name", ""),
                    "major": row.get("Major", ""),
                    "graduation_year": str(row.get("Graduation Year", "")),
                    "resume_link": row.get("Resume Link", ""),
                    "linkedin": row.get("Linkedin Link", ""),
                    "github": row.get("Github Link", ""),
                }
                break
        else:
            rec.setdefault("metadata", {})

    matched = sum(1 for r in records if r.get("metadata"))
    print(f"Enriched {matched} / {len(records)} DS3 records with members.csv metadata")
    return records


if members_df is not None:
    ds3_records = enrich_ds3_records(ds3_records, members_df)

## 5 — Combine all sources & save

In [ ]:
all_records = ds3_records + discord_records + reddit_records + kaggle_records

print("=" * 60)
print(f"TOTAL RESUMES EXTRACTED: {len(all_records)}")
print("=" * 60)

source_counts = {}
for r in all_records:
    source_counts[r["source"]] = source_counts.get(r["source"], 0) + 1

for src, count in sorted(source_counts.items()):
    print(f"  {src:20s} : {count}")

word_counts = [r["word_count"] for r in all_records]
if word_counts:
    print(f"\nWord count stats:")
    print(f"  min  = {min(word_counts)}")
    print(f"  mean = {np.mean(word_counts):.0f}")
    print(f"  max  = {max(word_counts)}")

In [ ]:
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(all_records, f, indent=2, ensure_ascii=False)

print(f"Saved {len(all_records)} records to {OUTPUT_PATH}")
print(f"File size: {OUTPUT_PATH.stat().st_size / 1024:.1f} KB")

## 6 — Quick sanity check

Preview a few records to make sure extraction looks reasonable.

In [ ]:
for i, rec in enumerate(all_records[:3]):
    print(f"\n{'=' * 60}")
    print(f"[{i}] {rec['filename']}  (source={rec['source']}, words={rec['word_count']})")
    print("-" * 60)
    print(rec["text"][:500])
    print("...")